In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

**Pragmatic inference** tests understanding of speaker intent beyond literal meaning, grounded in Grice's (1975) cooperative principle. Categories: scalar implicature ("some" → "not all"), indirect requests, irony, understatement, and relevance implicature.

**Known finding:** Gemini 2.5 Flash shows literal bias on scalar implicature — interpreting "some" logically rather than pragmatically.


## Interpreting the Score

Score = intended_accuracy - 0.1 × literal_trap_rate. Penalizes models that give literal interpretations where pragmatic meaning is intended.


### References
Grice (1975), Horn (1984), Searle (1975)


# Pragmatic Inference Benchmark

Tests understanding of speaker intent beyond literal meaning.
Covers scalar implicature, indirect requests, irony, and understatement.

**Cognitive Science**: Grice (1975), Horn (1984), Searle (1975)
**Human intended accuracy**: ~90-95%

In [ ]:
"""
Pragmatic inference test items.

Tests Gricean pragmatics: scalar implicature, indirect requests,
irony/understatement, and conversational maxim violations.

Based on Grice (1975) cooperative principle and conversational maxims:
- Quantity: Be as informative as required, not more
- Quality: Don't say what you believe to be false
- Relation: Be relevant
- Manner: Be clear, brief, orderly
"""

PRAGMATIC_ITEMS = [
    # ═══ SCALAR IMPLICATURE ═══
    # "Some" implicates "not all"
    {
        "id": "SI01",
        "type": "scalar_implicature",
        "context": "Teacher to parent: 'Some of the students passed the exam.'",
        "question": "What is the teacher implying?",
        "literal_meaning": "At least one student passed the exam.",
        "intended_meaning": "Not all students passed — some failed.",
        "intended_accept": ["not all", "some failed", "some did not pass", "not every", "some didn't"],
        "literal_accept": ["at least one", "one or more"],
    },
    {
        "id": "SI02",
        "type": "scalar_implicature",
        "context": "Restaurant reviewer: 'The food was warm.'",
        "question": "What is the reviewer suggesting about the food temperature?",
        "literal_meaning": "The food had a warm temperature.",
        "intended_meaning": "The food was not hot enough — it should have been hotter.",
        "intended_accept": ["not hot", "should have been hotter", "lukewarm", "wasn't hot", "not hot enough"],
        "literal_accept": ["warm temperature", "had warmth"],
    },
    {
        "id": "SI03",
        "type": "scalar_implicature",
        "context": "Job reference: 'The candidate is punctual and dresses neatly.'",
        "question": "What might this reference letter be implying by mentioning only these traits?",
        "literal_meaning": "The candidate arrives on time and dresses well.",
        "intended_meaning": "The candidate lacks more important qualities — this is damning with faint praise.",
        "intended_accept": ["faint praise", "lacking", "not much else", "not very competent", "nothing else positive", "damning", "doesn't have other", "not impressive"],
        "literal_accept": ["punctual", "dresses well", "on time"],
    },
    {
        "id": "SI04",
        "type": "scalar_implicature",
        "context": "A: 'Did everyone enjoy the party?' B: 'Some people had a good time.'",
        "question": "What is B implying?",
        "literal_meaning": "At least some people enjoyed it.",
        "intended_meaning": "Not everyone enjoyed the party — some didn't have a good time.",
        "intended_accept": ["not everyone", "some didn't", "not all", "some did not"],
        "literal_accept": ["some enjoyed", "at least some"],
    },
    {
        "id": "SI05",
        "type": "scalar_implicature",
        "context": "Professor commenting on a thesis: 'The bibliography is comprehensive.'",
        "question": "What might the professor be implying about the thesis overall?",
        "literal_meaning": "The bibliography is thorough and complete.",
        "intended_meaning": "The main content/argument of the thesis is weak — only the bibliography is noteworthy.",
        "intended_accept": ["thesis is weak", "content is weak", "only good thing", "nothing else", "rest is not", "faint praise", "damning"],
        "literal_accept": ["bibliography is good", "thorough"],
    },
    
    # ═══ INDIRECT REQUESTS ═══
    {
        "id": "IR01",
        "type": "indirect_request",
        "context": "One office worker to another: 'It's really cold in here, isn't it?'",
        "question": "What is the speaker actually requesting?",
        "literal_meaning": "A comment about the room temperature.",
        "intended_meaning": "Could you close the window / turn up the heat?",
        "intended_accept": ["close the window", "turn up", "heat", "make it warmer", "do something about", "shut the window", "turn on the heat", "adjust the temperature"],
        "literal_accept": ["cold", "temperature observation"],
    },
    {
        "id": "IR02",
        "type": "indirect_request",
        "context": "Passenger to driver: 'Do you know what time it is?'",
        "question": "What is the passenger actually asking for?",
        "literal_meaning": "Whether the driver has knowledge of the current time.",
        "intended_meaning": "Please tell me the actual time.",
        "intended_accept": ["tell me the time", "what time", "tell the time", "state the time", "the actual time"],
        "literal_accept": ["whether they know", "knowledge of time"],
    },
    {
        "id": "IR03",
        "type": "indirect_request",
        "context": "Guest at dinner: 'This soup could use a little something.'",
        "question": "What is the guest indirectly requesting?",
        "literal_meaning": "The soup is missing a flavor element.",
        "intended_meaning": "Please pass the salt/seasoning.",
        "intended_accept": ["salt", "seasoning", "pass the salt", "add seasoning", "spice", "pepper"],
        "literal_accept": ["missing flavor", "needs something"],
    },
    {
        "id": "IR04",
        "type": "indirect_request",
        "context": "Parent to teenager whose music is very loud: 'I'm trying to read.'",
        "question": "What is the parent actually requesting?",
        "literal_meaning": "The parent is informing the teenager of their current activity.",
        "intended_meaning": "Turn down the music / be quieter.",
        "intended_accept": ["turn down", "lower the volume", "be quiet", "reduce", "music", "quieter", "noise"],
        "literal_accept": ["reading", "trying to read"],
    },
    {
        "id": "IR05",
        "type": "indirect_request",
        "context": "Colleague looking at a messy shared kitchen: 'I wonder whose turn it is to clean.'",
        "question": "What is the colleague actually communicating?",
        "literal_meaning": "Genuine curiosity about the cleaning schedule.",
        "intended_meaning": "Someone should clean the kitchen — probably the person being addressed.",
        "intended_accept": ["clean", "you should clean", "it's your turn", "clean up", "tidy", "someone needs to clean"],
        "literal_accept": ["schedule", "whose turn"],
    },
    
    # ═══ IRONY / SARCASM ═══
    {
        "id": "IS01",
        "type": "irony",
        "context": "After waiting 2 hours in the rain for a bus: 'Well, this has been a delightful afternoon!'",
        "question": "What does the speaker actually mean?",
        "literal_meaning": "The afternoon has been pleasant and enjoyable.",
        "intended_meaning": "The afternoon has been terrible/miserable.",
        "intended_accept": ["terrible", "miserable", "awful", "horrible", "bad", "unpleasant", "not delightful", "opposite", "sarcastic", "sarcasm"],
        "literal_accept": ["delightful", "pleasant", "enjoyable"],
    },
    {
        "id": "IS02",
        "type": "irony",
        "context": "Student who received an F on a test: 'Another academic triumph!'",
        "question": "What does the student actually mean?",
        "literal_meaning": "The student achieved an academic success.",
        "intended_meaning": "The student failed badly — this is sarcastic self-deprecation.",
        "intended_accept": ["failed", "did badly", "sarcas", "opposite", "not a triumph", "ironic", "poor performance"],
        "literal_accept": ["triumph", "success"],
    },
    {
        "id": "IS03",
        "type": "irony",
        "context": "After a friend drops and breaks a plate: 'Smooth move, Grace Kelly!'",
        "question": "What is the speaker implying?",
        "literal_meaning": "Comparing the friend to the elegant Grace Kelly as a compliment.",
        "intended_meaning": "The friend was clumsy — the opposite of graceful.",
        "intended_accept": ["clumsy", "not graceful", "sarcas", "ironic", "opposite", "awkward", "ungraceful"],
        "literal_accept": ["graceful", "elegant", "compliment"],
    },
    {
        "id": "IS04",
        "type": "irony",
        "context": "Looking at a tiny, cramped apartment: 'What a palace! The king would be jealous.'",
        "question": "What is the speaker actually expressing?",
        "literal_meaning": "The apartment is grand and luxurious like a palace.",
        "intended_meaning": "The apartment is very small and unimpressive — the opposite of a palace.",
        "intended_accept": ["small", "tiny", "cramped", "unimpressive", "not a palace", "sarcas", "ironic", "opposite"],
        "literal_accept": ["palace", "grand", "luxurious"],
    },
    {
        "id": "IS05",
        "type": "irony",
        "context": "After a team loses 10-0: 'Well, we really showed them!'",
        "question": "What does the speaker actually mean?",
        "literal_meaning": "The team demonstrated their superiority.",
        "intended_meaning": "The team was badly defeated — this is sarcastic.",
        "intended_accept": ["lost badly", "defeated", "sarcas", "ironic", "opposite", "did terribly", "crushed"],
        "literal_accept": ["showed them", "won", "dominated"],
    },
    
    # ═══ UNDERSTATEMENT ═══
    {
        "id": "US01",
        "type": "understatement",
        "context": "News reporter about a category 5 hurricane: 'There might be a bit of wind later today.'",
        "question": "What is the reporter understating?",
        "literal_meaning": "Some mild wind is expected.",
        "intended_meaning": "A very severe/dangerous storm is coming — 'a bit of wind' massively understates the threat.",
        "intended_accept": ["severe", "dangerous", "hurricane", "major storm", "understat", "much worse", "extreme"],
        "literal_accept": ["some wind", "mild wind"],
    },
    {
        "id": "US02",
        "type": "understatement",
        "context": "Billionaire describing their wealth: 'I've done alright for myself.'",
        "question": "What is being understated?",
        "literal_meaning": "The person has had moderate financial success.",
        "intended_meaning": "The person is extremely wealthy — far beyond 'alright.'",
        "intended_accept": ["extremely wealthy", "very rich", "billionaire", "understat", "much more than alright", "enormous wealth"],
        "literal_accept": ["moderate success", "done okay"],
    },
    {
        "id": "US03",
        "type": "understatement",
        "context": "After running a marathon in record time: 'I suppose I'm in decent shape.'",
        "question": "What is the speaker understating?",
        "literal_meaning": "They are in reasonably good physical condition.",
        "intended_meaning": "They are in exceptional/elite physical condition — a marathon record is far beyond 'decent.'",
        "intended_accept": ["exceptional", "elite", "excellent", "extraordinary", "understat", "much better than decent", "outstanding"],
        "literal_accept": ["decent shape", "reasonable"],
    },
    {
        "id": "US04",
        "type": "understatement",
        "context": "Surgeon after a 14-hour operation that saved a patient's life: 'It wasn't the easiest day at work.'",
        "question": "What is the surgeon understating?",
        "literal_meaning": "The day at work was somewhat difficult.",
        "intended_meaning": "The day was extremely challenging and stressful — a 14-hour life-saving surgery is far beyond 'not easy.'",
        "intended_accept": ["extremely", "very difficult", "incredibly", "exhausting", "grueling", "understat", "much harder"],
        "literal_accept": ["somewhat difficult", "not easy"],
    },
    {
        "id": "US05",
        "type": "understatement",
        "context": "Astronaut describing their first spacewalk: 'The view was not bad.'",
        "question": "What is the astronaut understating?",
        "literal_meaning": "The view was acceptable or okay.",
        "intended_meaning": "The view was spectacular/breathtaking — seeing Earth from space is far beyond 'not bad.'",
        "intended_accept": ["spectacular", "breathtaking", "amazing", "incredible", "extraordinary", "understat", "much more than not bad", "stunning"],
        "literal_accept": ["not bad", "acceptable", "okay"],
    },
    # ═══ RELEVANCE IMPLICATURE ═══
    # Seemingly irrelevant response carries an implied meaning
    {
        "id": "RI01",
        "type": "relevance_implicature",
        "context": "A: 'Should we invite Tom to the party?' B: 'Well, he did break your favorite vase last time.'" ,
        "question": "What is B implying?",
        "literal_meaning": "Tom broke a vase at a previous event.",
        "intended_meaning": "No, we probably shouldn't invite Tom — he caused problems last time.",
        "intended_accept": ["shouldn't invite", "no", "don't invite", "bad idea", "against inviting", "not a good idea"],
        "literal_accept": ["broke", "vase"],
    },
    {
        "id": "RI02",
        "type": "relevance_implicature",
        "context": "A: 'How's the new restaurant downtown?' B: 'Well, the parking is convenient.'",
        "question": "What is B implying about the restaurant?",
        "literal_meaning": "The parking near the restaurant is good.",
        "intended_meaning": "The food or restaurant itself isn't great — the only positive thing B can say is about the parking.",
        "intended_accept": ["not great", "food is bad", "not good", "mediocre", "disappointing", "nothing else good", "only good thing"],
        "literal_accept": ["parking", "convenient"],
    },
    {
        "id": "RI03",
        "type": "relevance_implicature",
        "context": "A: 'Is Sarah a good singer?' B: 'She has a lovely stage presence.'",
        "question": "What is B implying about Sarah's singing?",
        "literal_meaning": "Sarah is engaging to watch on stage.",
        "intended_meaning": "Sarah's singing isn't good — B avoids commenting on it by praising something else.",
        "intended_accept": ["not good", "bad singer", "can't sing", "poor", "avoiding", "not a good singer", "weak"],
        "literal_accept": ["stage presence", "lovely"],
    },
    {
        "id": "RI04",
        "type": "relevance_implicature",
        "context": "A: 'Did you enjoy the movie?' B: 'The popcorn was fantastic.'",
        "question": "What is B implying about the movie?",
        "literal_meaning": "The popcorn at the cinema was very good.",
        "intended_meaning": "The movie wasn't good — the best thing about the experience was the popcorn.",
        "intended_accept": ["didn't enjoy", "movie was bad", "not good", "didn't like", "disappointing", "only good thing"],
        "literal_accept": ["popcorn", "fantastic"],
    },
    {
        "id": "RI05",
        "type": "relevance_implicature",
        "context": "A: 'Would you recommend this book?' B: 'It makes an excellent doorstop.'",
        "question": "What is B implying about the book?",
        "literal_meaning": "The book is heavy enough to prop open a door.",
        "intended_meaning": "The book is terrible — its only value is as a physical object, not for reading.",
        "intended_accept": ["terrible", "bad book", "not worth reading", "awful", "wouldn't recommend", "no good"],
        "literal_accept": ["doorstop", "heavy"],
    },
]


In [ ]:
"""
Social Cognition Benchmark 2: Pragmatic Inference

Tests understanding of pragmatic meaning — what speakers intend vs. what
they literally say. Covers Gricean maxims and implicatures.

Cognitive Science Basis:
- Grice (1975): Cooperative principle and conversational maxims
- Scalar implicature: "some" → "not all" (Horn, 1984)
- Indirect speech acts (Searle, 1975)
- Irony comprehension requires theory of mind + context integration

Categories tested:
1. Scalar implicature (5 items)
2. Indirect requests (5 items)
3. Irony/sarcasm (5 items)
4. Understatement (5 items)
5. Relevance implicature (5 items)

Metrics:
- Intended meaning accuracy: model identifies the speaker's actual intent
- Literal trap rate: how often model gives the literal (surface) interpretation
- Category breakdown: performance by pragmatic type

Score = intended_accuracy (primary) - 0.1 * literal_trap_rate (penalty)

Shortcut Resistance:
- Each item has both literal and intended meaning — model must choose
- Diverse pragmatic types prevent single-strategy shortcuts
- Rich context makes simple pattern matching insufficient
- Intended meanings require world knowledge + social reasoning
"""

import kaggle_benchmarks as kbench
import json as _json
def _safe_log(data): print(_json.dumps(data, indent=2, default=str))
from dataclasses import dataclass
import numpy as np
# Data defined above


@dataclass
class PragmaticResponse:
    """Model's pragmatic inference response."""
    speaker_intent: str   # What the speaker actually means/intends
    is_literal: bool      # Is the statement meant literally or figuratively?
    reasoning: str        # Brief explanation


def check_patterns(text: str, patterns: list) -> bool:
    """Check if text matches any pattern."""
    text_lower = text.lower()
    return any(p.lower() in text_lower for p in patterns)


@kbench.task(name="Pragmatic Inference")
def social_cog_pragmatic(llm) -> float:
    """Pragmatic Inference: Tests understanding of speaker intent beyond literal meaning, including scalar implicature, indirect requests, and conversational maxim violations.
    """
    results = []
    category_results = {}
    
    for item in PRAGMATIC_ITEMS:
        cat = item["type"]
        if cat not in category_results:
            category_results[cat] = {"intended": 0, "literal": 0, "total": 0}
        
        prompt = (
            f"Read this situation carefully:\n\n"
            f"Context: {item['context']}\n\n"
            f"Question: {item['question']}\n\n"
            f"Consider both the literal meaning and what the speaker actually intends "
            f"to communicate. What is the speaker's TRUE intended meaning?"
        )
        
        with kbench.chats.new(f"pragmatic_{item['id']}"):
            try:
                response = llm(prompt, response_format=PragmaticResponse)
                speaker_intent = response.speaker_intent
                is_literal = response.is_literal
            except Exception:
                raw = llm(prompt)
                speaker_intent = raw
                is_literal = False
        
        got_intended = check_patterns(speaker_intent, item["intended_accept"])
        got_literal = check_patterns(speaker_intent, item["literal_accept"])
        
        # If model gives both intended and literal markers, count intended
        if got_intended:
            got_literal = False
        
        results.append({
            "id": item["id"],
            "type": cat,
            "got_intended": got_intended,
            "got_literal": got_literal,
            "model_answer": speaker_intent,
            "marked_literal": is_literal,
        })
        
        category_results[cat]["total"] += 1
        if got_intended:
            category_results[cat]["intended"] += 1
        if got_literal:
            category_results[cat]["literal"] += 1
    
    # ── Compute Metrics ──
    
    intended_acc = sum(1 for r in results if r["got_intended"]) / len(results)
    literal_trap = sum(1 for r in results if r["got_literal"]) / len(results)
    
    # Category breakdown
    cat_scores = {}
    for cat, data in category_results.items():
        cat_scores[cat] = {
            "intended_accuracy": round(data["intended"] / max(data["total"], 1), 4),
            "literal_trap_rate": round(data["literal"] / max(data["total"], 1), 4),
            "count": data["total"],
        }
    
    # ── Composite Score ──
    score = intended_acc - 0.1 * literal_trap
    score = round(float(np.clip(score, 0, 1)), 4)
    
    _safe_log({
        "benchmark": "Pragmatic Inference",
        "n_items": len(results),
        "intended_accuracy": round(intended_acc, 4),
        "literal_trap_rate": round(literal_trap, 4),
        "composite_score": score,
        "by_category": cat_scores,
        "per_item": results,
    })
    
    return score

social_cog_pragmatic.run(llm=kbench.llm)

